# Train-Test-Validation Split 

This notebook performs a split of the financial fraud dataset into training, validation, and test sets.


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Load Dataset

In [2]:
import pandas as pd
import os

path = "../data/raw/financial-fraud-detection-dataset"
csv_file = os.path.join(path, "Synthetic_Financial_datasets_log.csv")
print(f"Looking for file at: {csv_file}")


df = pd.read_csv(csv_file)
print("Shape:", df.shape)
print(df.head())
print(df.info())

Looking for file at: ../data/raw/financial-fraud-detection-dataset/Synthetic_Financial_datasets_log.csv
Shape: (6362620, 11)
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        

In [3]:
# Define features (X) and target (y)
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# First split: 70% for training, 30% for temporary (validation + test)
# Stratify by y to maintain fraud rate proportion
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: split the 30% temporary set into 50% validation and 50% test
# This results in 15% validation and 15% test of the original data
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Recreate dataframes
train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)


print("\n=== STRATIFIED SPLITS ===")
print(f"\nTraining set:")
print(f"  Size: {len(train_df):,} ({len(train_df)/len(df)*100:.2f}%)")
print(f"  Fraud rate: {train_df['isFraud'].mean():.4f}")

print(f"\nValidation set:")
print(f"  Size: {len(val_df):,} ({len(val_df)/len(df)*100:.2f}%)")
print(f"  Fraud rate: {val_df['isFraud'].mean():.4f}")

print(f"\nTest set:")
print(f"  Size: {len(test_df):,} ({len(test_df)/len(df)*100:.2f}%)")
print(f"  Fraud rate: {test_df['isFraud'].mean():.4f}")

# Check consistency of fraud rates
print("\n=== FRAUD RATE CONSISTENCY ===")
fraud_rates = [
    train_df['isFraud'].mean(),
    val_df['isFraud'].mean(),
    test_df['isFraud'].mean()
]
print(f"Fraud rates: Train={fraud_rates[0]:.4f}, Val={fraud_rates[1]:.4f}, Test={fraud_rates[2]:.4f}")
print(f"Overall dataset fraud rate: {df['isFraud'].mean():.4f}")
print(f"Std deviation across splits: {np.std(fraud_rates):.6f}")


=== STRATIFIED SPLITS ===

Training set:
  Size: 4,453,834 (70.00%)
  Fraud rate: 0.0013

Validation set:
  Size: 954,393 (15.00%)
  Fraud rate: 0.0013

Test set:
  Size: 954,393 (15.00%)
  Fraud rate: 0.0013

=== FRAUD RATE CONSISTENCY ===
Fraud rates: Train=0.0013, Val=0.0013, Test=0.0013
Overall dataset fraud rate: 0.0013
Std deviation across splits: 0.000000


## Stratified Train-Test-Validation Split

We will perform a stratified split to ensure that the proportion of fraudulent transactions is similar across the training, validation, and test sets. This is crucial for imbalanced datasets.

We will split the data into:
- 70% Training
- 15% Validation
- 15% Test

In [4]:
# Save fraud-balanced splits
balanced_split_dir = "../data/splits"
os.makedirs(balanced_split_dir, exist_ok=True)

train_df.to_csv(os.path.join(balanced_split_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(balanced_split_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(balanced_split_dir, "test.csv"), index=False)

print(f"✅ Saved fraud-balanced splits to: {balanced_split_dir}")
print(f"\nFiles saved:")
print(f"  - train.csv: {len(train_df):,} samples")
print(f"  - val.csv: {len(val_df):,} samples")
print(f"  - test.csv: {len(test_df):,} samples")
print(f"\n⚠️ These splits have consistent fraud rates but are still highly imbalanced!")

print(f"Use resampling techniques below to balance the TRAINING set only.")

✅ Saved fraud-balanced splits to: ../data/splits

Files saved:
  - train.csv: 4,453,834 samples
  - val.csv: 954,393 samples
  - test.csv: 954,393 samples

⚠️ These splits have consistent fraud rates but are still highly imbalanced!
Use resampling techniques below to balance the TRAINING set only.


In [5]:
# Check transaction type distribution
print("\nTransaction type distribution:")
print("\nTraining set:")
print(train_df['type'].value_counts(normalize=True))
print("\nValidation set:")
print(val_df['type'].value_counts(normalize=True))
print("\nTest set:")
print(test_df['type'].value_counts(normalize=True))


Transaction type distribution:

Training set:
type
CASH_OUT    0.351591
PAYMENT     0.338170
CASH_IN     0.219979
TRANSFER    0.083758
DEBIT       0.006502
Name: proportion, dtype: float64

Validation set:
type
CASH_OUT    0.351669
PAYMENT     0.337463
CASH_IN     0.219925
TRANSFER    0.084351
DEBIT       0.006593
Name: proportion, dtype: float64

Test set:
type
CASH_OUT    0.351998
PAYMENT     0.338716
CASH_IN     0.219657
TRANSFER    0.083153
DEBIT       0.006476
Name: proportion, dtype: float64


## Summary Statistics

In [6]:
# Create a summary dataframe
summary_data = {
    'Dataset': ['Training', 'Validation', 'Test', 'Total'],
    'Size': [len(train_df), len(val_df), len(test_df), len(df)],
    'Fraud Count': [train_df['isFraud'].sum(), val_df['isFraud'].sum(), test_df['isFraud'].sum(), df['isFraud'].sum()],
    'Fraud Rate': [train_df['isFraud'].mean(), val_df['isFraud'].mean(), test_df['isFraud'].mean(), df['isFraud'].mean()]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Size %'] = (summary_df['Size'] / len(df) * 100).round(2)
summary_df['Fraud Rate'] = summary_df['Fraud Rate'].round(4)

print("\n=== SUMMARY ===")
print(summary_df.to_string(index=False))

# Save summary
summary_path = os.path.join(balanced_split_dir, "split_summary.csv")

summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")


=== SUMMARY ===
   Dataset    Size  Fraud Count  Fraud Rate  Size %
  Training 4453834         5749      0.0013    70.0
Validation  954393         1232      0.0013    15.0
      Test  954393         1232      0.0013    15.0
     Total 6362620         8213      0.0013   100.0

Summary saved to: ../data/splits/split_summary.csv
